<style>
:root { --jp-content-font-size1: 11px; --jp-code-font-size: 10px; --jp-ui-font-size1: 10px; }
body { font-size: 11px; line-height: 1.4; }
h1 { font-size: 20px !important; }
h2 { font-size: 16px !important; margin-top: 1.2em !important; }
h3 { font-size: 13px !important; }
table { font-size: 9px !important; table-layout: auto !important; }
th, td { font-size: 9px !important; padding: 2px 4px !important; }
.jp-Cell { page-break-inside: avoid; }
.page-break { page-break-before: always; }
pre { font-size: 9px !important; line-height: 1.3 !important; }
@page { margin: 0.6in 0.5in; }
</style>

# Embeddings: Embeddings & Similarity in FFAI's RAG System

This notebook demonstrates the embeddings module from FFAI's RAG system.
All API calls are **mocked** — no API keys required.
Replace mocks with real calls in production.

<div class="page-break"></div>

---

## Section 1: Setup

In [ ]:
import sys
from pathlib import Path

# Walk up from CWD to find project root (has pyproject.toml)
_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

from unittest.mock import MagicMock, patch  # noqa: E402

from ffai.rag.embed import Embeddings  # noqa: E402

print(f'Project root: {_project_root}')
print('Embeddings imported successfully')

<div class="page-break"></div>

---

## Section 2: Embeddings Overview

Embeddings supports two embedding backends:
- **API path**: Uses LiteLLM to call provider APIs (Mistral, OpenAI, Azure, etc.)
- **Local path**: Uses `sentence-transformers` for zero-cost local inference

Key features include LRU embedding caching and automatic provider detection.

In [ ]:
print(Embeddings.__doc__)

<div class="page-break"></div>

---

## Section 3: API Embeddings (Mocked)

We mock the `embedding` function imported by the module to return fake
1024-dimensional vectors. This demonstrates the full API surface without
needing real credentials.

In [ ]:
DUMMY_DIM = 1024

async def _fake_embedding_response(*args, **kwargs):
    texts = kwargs.get('input', args[1] if len(args) > 1 else [])
    if isinstance(texts, str):
        texts = [texts]
    mock_response = MagicMock()
    mock_response.data = [
        {"index": i, "embedding": [0.1 * (i + 1)] * DUMMY_DIM}
        for i in range(len(texts))
    ]
    return mock_response

_embed_patcher = patch(
    'litellm.aembedding',
    side_effect=_fake_embedding_response,
)
_mock_litellm = _embed_patcher.start()

emb = Embeddings(model='mistral/mistral-embed', api_key='fake-key')
print(f'Model: {emb.model}')
print(f'Provider: {emb.provider}')
print(f'Is local: {emb.is_local}')

In [ ]:
vectors = emb.embed(['Hello world', 'Another text'])
print(f'Number of vectors: {len(vectors)}')
print(f'Dimension of first vector: {len(vectors[0])}')
print(f'First 5 values of vector 0: {vectors[0][:5]}')
print(f'First 5 values of vector 1: {vectors[1][:5]}')

In [ ]:
single = emb.embed_single('test')
print(f'Single embedding length: {len(single)}')
print(f'First 5 values: {single[:5]}')

In [ ]:
dim = len(vectors[0])
print(f'Embedding dimension: {dim}')

<div class="page-break"></div>

---

## Section 4: LRU Cache Behavior

Embeddings caches embedding results in an `OrderedDict`-based LRU cache.
Repeated calls for the same text are served from cache — no duplicate API calls.

In [ ]:
_mock_litellm.reset_mock()
emb.clear_cache()

v1 = emb.embed_single('test query')
v2 = emb.embed_single('test query')

print(f'Vectors are identical: {v1 == v2}')
print(f'litellm.embedding call count: {_mock_litellm.call_count}')
print('(Expected 1 — second call served from cache)')

In [ ]:
stats_before = emb.cache_stats()
print(f'Cache stats before: {stats_before}')

emb.embed_single('new text')
stats_after = emb.cache_stats()
print(f'Cache stats after:  {stats_after}')

In [ ]:
cleared = emb.clear_cache()
print(f'Entries cleared: {cleared}')
print(f'Cache after clear: {emb.cache_stats()}')

<div class="page-break"></div>

---

## Section 5: Cosine Similarity

`Embeddings.cosine_similarity()` is a static method computing cosine
similarity between two vectors. Scores range from -1 (opposite) to 1 (identical).

In [ ]:
# Identical vectors -> similarity = 1.0
identical = [1.0, 2.0, 3.0, 4.0]
sim_identical = Embeddings.cosine_similarity(identical, identical)
print(f'Identical vectors:   {sim_identical:.6f}')

# Orthogonal vectors -> similarity ~ 0.0
orth_a = [1.0, 0.0, 0.0, 0.0]
orth_b = [0.0, 1.0, 0.0, 0.0]
sim_orthogonal = Embeddings.cosine_similarity(orth_a, orth_b)
print(f'Orthogonal vectors:  {sim_orthogonal:.6f}')

# Similar vectors -> high similarity
sim_a = [1.0, 2.0, 3.0, 4.0]
sim_b = [1.1, 2.1, 3.1, 4.1]
sim_high = Embeddings.cosine_similarity(sim_a, sim_b)
print(f'Similar vectors:     {sim_high:.6f}')

# Opposite vectors -> similarity ~ -1.0
opp_a = [1.0, 0.0]
opp_b = [-1.0, 0.0]
sim_opposite = Embeddings.cosine_similarity(opp_a, opp_b)
print(f'Opposite vectors:    {sim_opposite:.6f}')

<div class="page-break"></div>

---

## Section 6: Provider Detection

Embeddings auto-detects the provider from the model string prefix.

In [ ]:
models = [
    'mistral/mistral-embed',
    'openai/text-embedding-3-small',
    'local/all-MiniLM-L6-v2',
]

for m in models:
    if m.startswith('local/'):
        provider = 'local'
        is_local = True
    else:
        tmp = Embeddings(model=m, api_key='fake-key')
        provider = tmp.provider
        is_local = tmp.is_local
    print(f'Model: {m:40s}  provider={provider:10s}  is_local={is_local}')

<div class="page-break"></div>

---

## Section 7: Practical Use Case — Pre-screening

A common RAG pattern: embed documents and a query, then rank documents by
cosine similarity to the query. This pre-screens which documents to send to
the LLM, reducing token cost and improving relevance.

In [ ]:
documents = [
    'FFAI provides financial data APIs for institutional investors.',
    'The weather in London is rainy and cold today.',
    'Embedding models convert text into numerical vectors.',
    'Revenue grew 15% year-over-year in Q3 2024.',
    'Python is a popular programming language for data science.',
]
query = 'financial performance and revenue data'

async def _contextual_fake_embedding(*args, **kwargs):
    texts = kwargs.get('input', args[1] if len(args) > 1 else [])
    if isinstance(texts, str):
        texts = [texts]
    mock_response = MagicMock()
    mock_response.data = [
        {
            'index': i,
            'embedding': [0.5 + 0.1 * (hash(t) % 10) if j == 0
                         else 0.01 * (j % 5) for j in range(DUMMY_DIM)],
        }
        for i, t in enumerate(texts)
    ]
    return mock_response

_embed_patcher.stop()
_ctx_patcher = patch(
    'litellm.aembedding',
    side_effect=_contextual_fake_embedding,
)
_ctx_patcher.start()

emb2 = Embeddings(model='mistral/mistral-embed', api_key='fake-key')

doc_vectors = emb2.embed(documents)
query_vector = emb2.embed_single(query)

scores = [
    (i, Embeddings.cosine_similarity(query_vector, dv))
    for i, dv in enumerate(doc_vectors)
]
scores.sort(key=lambda x: x[1], reverse=True)

print('Documents ranked by similarity to query:')
print(f'  Query: "{query}"')
print()
for rank, (idx, score) in enumerate(scores, 1):
    print(f'  {rank}. [score={score:.4f}] {documents[idx]}')

---

This concludes the embeddings module demonstration. In production, replace
the mocked `litellm.embedding` with real API credentials to get meaningful
similarity scores.